# LLM-as-a-Judge: Automated Answer Quality

`03-rag-evals.ipynb` produced 395 `(answer_llm, answer_orig)` pairs, but nothing has actually scored whether they mean the same thing — comparing 395 answer pairs by eye isn't realistic. Two answers can be phrased completely differently and still be correct, so exact string matching won't work either. This notebook uses an LLM as the judge: given a question, the ground-truth FAQ answer, and the RAG-generated answer, it decides whether they're semantically equivalent and returns a `good`/`bad` verdict with reasoning.

## 1. Defining the Judge's Output Schema

`AnswerEvaluation` forces a structured verdict: `reasoning` (a free-text justification, asked for first so the model reasons before committing to a score) and `score` (`Literal["good", "bad"]`, and only those two values).

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

## 2. The Judge Prompt

`aqa_judge_instructions` sets fair ground rules for the comparison: the AI answer doesn't need to match word-for-word, extra detail is fine, and it should only be marked `bad` if it's wrong or misses the key point — otherwise a stricter judge would penalize correct answers just for being phrased differently. `aqa_judge_prompt` is a template that slots in the question plus both answers for a single record.

In [ ]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [ ]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

## 3. Setup: Client and RAG Answers

Standard OpenAI client setup, plus the helpers from `zoomcamp/evaluation_utils.py` this notebook needs: `calc_price`, `calc_total_price`, `llm_structured_retry`, and `map_progress`. `data/rag-answers-new.csv` — the output of `03-rag-evals.ipynb` — is loaded as the set of records to judge.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from zoomcamp.evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [ ]:
import pandas as pd

df_answers = pd.read_csv("./data/rag-answers.csv")
answers = df_answers.to_dict(orient="records")

In [ ]:
rec = answers[0]
rec

## 4. Judging a Single Answer

Format the prompt for the first record, then call `llm_structured_retry` with `AnswerEvaluation` as the `output_type` — same structured-output pattern used for question generation in `01-data-gen.ipynb`, just with a different schema and prompt. The judge correctly marks a reworded-but-equivalent answer as `good`, with reasoning that explains why. Cost per verdict: a fraction of a cent (`calc_price(usage)`).

In [ ]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

In [ ]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

In [ ]:
calc_price(usage)

## 5. Wrapping the Judge in a Reusable Function

`evaluate_aqa(question, answer_orig, answer_llm, model=...)` bundles prompt formatting + the retrying LLM call into one function. `judge_record(rec)` builds on top of it, returning a clean result dict (`question`, `document`, `score`, `reasoning`) alongside the usage — mirroring how `generate_ground_truth` and `generate_rag_answer` were structured in the earlier notebooks.

In [ ]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

In [ ]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

## 6. Judging All 395 Answers in Parallel

Same `ThreadPoolExecutor` + `map_progress` pattern used throughout this module, now judging every RAG answer against its ground-truth counterpart. Judging all 395 pairs costs 0.251 total (`calc_total_price`) — noticeably more than generating the questions ($0.057), since each judge call reads a full question + two answers as input.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

In [ ]:
results[10]

In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
calc_total_price(usages)

## 7. Results: How Good Is the RAG Pipeline?

`df_eval.score.value_counts()` gives the final verdict: **379 good / 16 bad**, i.e. **≈95.9% good / 4.1% bad**. Filtering to the `bad` rows surfaces the actual failure cases (e.g. GPU-hours questions, peer-review requirements, model-selection details) with the judge's reasoning attached — a concrete debugging list instead of a single opaque score. Results are saved to `data/rag-evaluations-new.csv`, completing the evaluation chain: ground truth → search metrics → RAG answers → judged quality.

In [ ]:
df_eval = pd.DataFrame(evaluations)

In [ ]:
df_eval.head()

In [ ]:
df_eval.score.value_counts()

In [ ]:
df_eval.score.value_counts(normalize=True)

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

In [ ]:
df_eval.to_csv("./data/rag-evaluations.csv", index=False)